# Dashboard Data Preparation
Genera los CSV de conteos que se usan en el dashboard (Stack Overflow Developer Survey).

Pestañas: **Uso actual**, **Tendencias futuras**, **Demografía**.

In [1]:
import pandas as pd
import os

In [2]:
!wget -O survey-data.csv https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv

--2026-09-30 18:19:44--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104
HTTP request sent, awaiting response... 200 OK
Length: 159525875 (152M) [text/csv]
Saving to: ‘survey-data.csv’

survey-data.csv     100%[===================>] 152.13M  3.85MB/s    in 46s     

2026-09-30 18:20:30 (3.33 MB/s) - ‘survey-data.csv’ saved [159525875/159525875]


In [3]:
df = pd.read_csv('survey-data.csv')
df.shape

(65437, 114)

## Columnas con varios valores (separados por `;`)
Se separan con `split` + `explode` y se cuentan.

In [4]:
os.makedirs('dashboard-data', exist_ok=True)

def multi_counts(col, top=None):
    s = df[col].dropna().str.split(';').explode().str.strip()
    out = s.value_counts().reset_index()
    out.columns = [col, 'Count']
    return out.head(top) if top else out

tech = {
    # Pestaña 1: Uso actual
    'LanguageHaveWorkedWith': 10,
    'DatabaseHaveWorkedWith': 10,
    'PlatformHaveWorkedWith': None,   # todas las plataformas
    'WebframeHaveWorkedWith': 10,
    # Pestaña 2: Tendencias futuras
    'LanguageWantToWorkWith': 10,
    'DatabaseWantToWorkWith': 10,
    'PlatformWantToWorkWith': None,
    'WebframeWantToWorkWith': 10,
}

for col, top in tech.items():
    out = multi_counts(col, top)
    out.to_csv(f'dashboard-data/{col}.csv', index=False)
    print(col, '->', len(out), 'filas')

LanguageHaveWorkedWith -> 10 filasDatabaseHaveWorkedWith -> 10 filasPlatformHaveWorkedWith -> 27 filasWebframeHaveWorkedWith -> 10 filasLanguageWantToWorkWith -> 10 filasDatabaseWantToWorkWith -> 10 filas
PlatformWantToWorkWith -> 27 filasWebframeWantToWorkWith -> 10 filas

## Pestaña 3: Demografía

In [5]:
# Países (top 20)
country = df['Country'].value_counts().head(20).reset_index()
country.columns = ['Country', 'Count']
country.to_csv('dashboard-data/Country.csv', index=False)

# Grupos de edad (en orden lógico)
age_order = ['Under 18 years old', '18-24 years old', '25-34 years old',
             '35-44 years old', '45-54 years old', '55-64 years old',
             '65 years or older', 'Prefer not to say']
age = df['Age'].value_counts().reindex(age_order).dropna().reset_index()
age.columns = ['Age', 'Count']
age['Count'] = age['Count'].astype(int)
age.to_csv('dashboard-data/Age.csv', index=False)

# Nivel educativo
ed = df['EdLevel'].value_counts().reset_index()
ed.columns = ['EdLevel', 'Count']
ed.to_csv('dashboard-data/EdLevel.csv', index=False)

print(sorted(os.listdir('dashboard-data')))

['Age.csv', 'Country.csv', 'DatabaseHaveWorkedWith.csv', 'DatabaseWantToWorkWith.csv', 'EdLevel.csv', 'LanguageHaveWorkedWith.csv', 'LanguageWantToWorkWith.csv', 'PlatformHaveWorkedWith.csv', 'PlatformWantToWorkWith.csv', 'WebframeHaveWorkedWith.csv', 'WebframeWantToWorkWith.csv']

## Vista previa rápida

In [6]:
for f in sorted(os.listdir('dashboard-data')):
    print('\n' + f)
    print(pd.read_csv(f'dashboard-data/{f}').head(10).to_string(index=False))


Age.csv               Age  Count
Under 18 years old   2568
   18-24 years old  14098
   25-34 years old  23911
   35-44 years old  14942
   45-54 years old   6249
   55-64 years old   2575
 65 years or older    772
 Prefer not to say    322

Country.csv
                                             Country  Count
                            United States of America  11095
                                             Germany   4947
                                               India   4231
United Kingdom of Great Britain and Northern Ireland   3224
                                             Ukraine   2672
                                              France   2110
                                              Canada   2104
                                              Poland   1534
                                         Netherlands   1449
                                              Brazil   1375

DatabaseHaveWorkedWith.csv
DatabaseHaveWorkedWith  Count
            PostgreSQL  255

## Empaquetar para descargar
Crea un solo `.zip` con los 11 CSV.

In [7]:
import shutil
shutil.make_archive('dashboard-data', 'zip', 'dashboard-data')
'dashboard-data.zip'

'dashboard-data.zip'